In [1]:
!pip install -q transformers torch scikit-learn tqdm pandas

In [13]:
MODEL_PATH = '/kaggle/input/datasets/smrubs/legalbert-predex/best_hierarchical_model.pt'
TEST_CSV   = '/kaggle/input/datasets/smrubs/legalbert-predex/predex_test_final.csv'

In [3]:
import torch
import transformers
transformers.logging.set_verbosity_error()

CHUNK_SIZE  = 512
STRIDE      = 256
MAX_CHUNKS  = 16
GRU_HIDDEN  = 256
GRU_LAYERS  = 2
DROPOUT     = 0.3
UNFREEZE_LAYERS = 3
MODEL_NAME  = "law-ai/InLegalBERT"
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
# ════════════════════════════════════════════════════
# CELL 4 — Model definition (identical to training)
# ════════════════════════════════════════════════════

import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class HierarchicalLegalBERT(nn.Module):
    def __init__(self, model_name, gru_hidden=256, gru_layers=2,
                 dropout=0.3, num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False
        bert_dim = self.bert.config.hidden_size  # 768
        self.gru = nn.GRU(
            input_size    = bert_dim,
            hidden_size   = gru_hidden,
            num_layers    = gru_layers,
            batch_first   = True,
            dropout       = dropout if gru_layers > 1 else 0,
            bidirectional = True
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask, chunk_counts):
        B, max_chunks, seq_len = input_ids.shape
        ids   = input_ids.view(B * max_chunks, seq_len)
        masks = attention_mask.view(B * max_chunks, seq_len)
        with torch.no_grad():
            bert_out = self.bert(input_ids=ids, attention_mask=masks)
        cls_emb = bert_out.last_hidden_state[:, 0, :]
        cls_emb = cls_emb.view(B, max_chunks, -1)
        cls_emb = self.dropout(cls_emb)
        packed = nn.utils.rnn.pack_padded_sequence(
            cls_emb, chunk_counts.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        final = torch.cat([hidden[-2], hidden[-1]], dim=1)
        final = self.dropout(final)
        return self.classifier(final)

In [14]:
# ════════════════════════════════════════════════════
# CELL 5 — Load model weights
# ════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = HierarchicalLegalBERT(MODEL_NAME, GRU_HIDDEN, GRU_LAYERS, DROPOUT)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

# Handle both raw state_dict and checkpoint dict formats
state_dict = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print("✓ Model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Model loaded successfully


In [15]:
# ════════════════════════════════════════════════════
# CELL 6 — Load & inspect test data
# ════════════════════════════════════════════════════

import pandas as pd

df = pd.read_csv(TEST_CSV)
print(f"Columns: {df.columns.tolist()}")
print(f"Shape  : {df.shape}")
print(df.head(3))

# predex CSV uses 'Input' as the case text and 'Label' as ground truth
# The 'Output' column contains expert rationale with a leading label prefix like "1[ds]..."
TEXT_COL  = 'Input'
LABEL_COL = 'Label'

# Drop rows missing text or label
df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
df[LABEL_COL] = df[LABEL_COL].astype(int)
print(f"\nUsable rows: {len(df)}  |  Label dist: {df[LABEL_COL].value_counts().to_dict()}")


Columns: ['Case Name', 'Input', 'Output', 'Label', 'Count', 'Decision_Count', 'word_count', 'court']
Shape  : (1356, 8)
                                     Case Name  \
0             M/S. Swil Ltd Vs. State Of Delhi   
1          N.S.Nagendra Vs. State Of Karnataka   
2  Mannu and Others Vs. State of Uttar Pradesh   

                                               Input  \
0  M.B. Shah, J. 1. Leave granted in S.L.P. (Crl....   
1  A.K. Sikri, J. 1. The petitioner is convicted ...   
2  KAILASAM, J. 1. This appeal by the 10 appellan...   

                                              Output  Label  Count  \
0  1[ds]6. In our view, from the facts stated abo...      1   1139   
1  0[ds]As per the report of the hand-writing exp...      0   1087   
2  0[ds]7. In appeal before us, the learned couns...      0   1441   

   Decision_Count  word_count court  
0             536         995  high  
1             377         967  high  
2             311        1321  high  

Usable rows: 1356  |

In [16]:
# ════════════════════════════════════════════════════
# CELL 7 — Chunking helper
# ════════════════════════════════════════════════════

def chunk_text(text, tokenizer, chunk_size=512, stride=256, max_chunks=8):
    enc      = tokenizer(str(text), add_special_tokens=False, return_tensors=None)
    all_ids  = enc['input_ids']
    all_mask = enc['attention_mask']
    effective = chunk_size - 2
    chunks, start = [], 0
    while start < len(all_ids):
        end        = start + effective
        chunk_ids  = [tokenizer.cls_token_id] + all_ids[start:end]  + [tokenizer.sep_token_id]
        chunk_mask = [1]                       + all_mask[start:end] + [1]
        pad        = chunk_size - len(chunk_ids)
        chunk_ids  += [tokenizer.pad_token_id] * pad
        chunk_mask += [0] * pad
        chunks.append((chunk_ids, chunk_mask))
        if end >= len(all_ids):
            break
        start += stride
    return chunks[:max_chunks]


In [17]:
# ════════════════════════════════════════════════════
# CELL 8 — Run inference
# ════════════════════════════════════════════════════
from tqdm import tqdm
import numpy as np

all_preds  = []
all_probs  = []
all_labels = df[LABEL_COL].tolist()

model.eval()
with torch.no_grad():
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Inference"):
        chunks = chunk_text(row[TEXT_COL], tokenizer, CHUNK_SIZE, STRIDE, MAX_CHUNKS)
        n = len(chunks)

        ids   = torch.tensor([c[0] for c in chunks], dtype=torch.long).unsqueeze(0).to(DEVICE)  # (1, n, 512)
        masks = torch.tensor([c[1] for c in chunks], dtype=torch.long).unsqueeze(0).to(DEVICE)
        counts = torch.tensor([n], dtype=torch.long)

        logits = model(ids, masks, counts)                          # (1, 2)
        probs  = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred   = int(logits.argmax(dim=-1).item())

        all_preds.append(pred)
        all_probs.append(probs[1])   # prob of class 1 (accepted)


Inference: 100%|██████████| 1356/1356 [07:18<00:00,  3.10it/s]


In [18]:
# ════════════════════════════════════════════════════
# CELL 9 — Metrics & results
# ════════════════════════════════════════════════════
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix,
                              roc_auc_score)

print("=" * 50)
print("  RESULTS ON PREDEX TEST SET")
print("=" * 50)
print(f"  Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"  Macro-F1  : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(all_labels, all_probs):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=['Rejected (0)', 'Accepted (1)']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


  RESULTS ON PREDEX TEST SET
  Accuracy  : 0.7146
  Macro-F1  : 0.7145
  ROC-AUC   : 0.8282

              precision    recall  f1-score   support

Rejected (0)       0.63      0.83      0.72       594
Accepted (1)       0.83      0.62      0.71       762

    accuracy                           0.71      1356
   macro avg       0.73      0.73      0.71      1356
weighted avg       0.74      0.71      0.71      1356

Confusion Matrix:
[[495  99]
 [288 474]]


In [19]:
# ════════════════════════════════════════════════════
# CELL 10 — Save predictions CSV
# ════════════════════════════════════════════════════
out_df = df.copy()
out_df['predicted_label'] = all_preds
out_df['prob_accepted']   = all_probs
out_df['correct']         = (out_df['predicted_label'] == out_df[LABEL_COL])

out_path = '/kaggle/working/predex_predictions.csv'
out_df.to_csv(out_path, index=False)
print(f"\n✓ Predictions saved to: {out_path}")



✓ Predictions saved to: /kaggle/working/predex_predictions.csv
